# Quantization-Aware Ternary Distillation (QAT-TD) on Kaggle GPU
## Complete Factorial Experiment Suite & Evaluation Benchmark

This notebook trains and benchmarks 4+1 model variations to isolate the impact of **Knowledge Distillation (KD)** and **1.58-bit Ternary Quantization-Aware Training (QAT)**:

1. **Model A (Baseline Student):** `SmolLM-135M` (FP16, CE only) — *Reference capacity*
2. **Model B (Distilled Student):** `SmolLM-360M` $\to$ `SmolLM-135M` (FP16, KD + CE) — *Distillation gain*
3. **Model C (Ternary QAT Student):** `SmolLM-135M` (1.58-bit Ternary QAT + STE, CE only) — *Quantization degradation*
4. **Model D (KD + Ternary QAT Student):** `SmolLM-360M` $\to$ `SmolLM-135M` (Ternary QAT, KD + CE) — ***Core baseline***
5. **Model E (XAI-KD + Ternary QAT):** `SmolLM-360M` $\to$ `SmolLM-135M` (Ternary QAT, XAI Layer-Weighted KD) — *Proposed extension inspired by DiXtill*

In [ ]:
# 1. Install & Import Required Libraries
!pip install -q transformers datasets accelerate tqdm matplotlib

import os, math, time, json
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Executing on: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Straight-Through Estimator & Ternary Linear Layer Definition
class TernarySTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, weight, threshold_factor):
        scale = weight.abs().mean().clamp_min(1e-8)
        threshold = threshold_factor * scale
        return torch.where(
            weight > threshold, scale,
            torch.where(weight < -threshold, -scale, torch.zeros_like(weight))
        )

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output, None

class TernaryLinear(nn.Module):
    def __init__(self, original_layer, threshold_factor=0.7):
        super().__init__()
        self.weight = original_layer.weight  # Latent FP master weight
        self.bias = original_layer.bias      # Preserved in full-precision
        self.threshold_factor = threshold_factor

    def forward(self, x):
        ternary_weight = TernarySTE.apply(self.weight, self.threshold_factor)
        return F.linear(x, ternary_weight, self.bias)

def replace_linear_layers(module, threshold_factor=0.7):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear):
            setattr(module, name, TernaryLinear(child, threshold_factor=threshold_factor))
        else:
            replace_linear_layers(child, threshold_factor=threshold_factor)

def compute_kd_loss(student_logits, teacher_logits, attention_mask, temperature=2.0):
    s_log = F.log_softmax(student_logits / temperature, dim=-1)
    t_prob = F.softmax(teacher_logits / temperature, dim=-1)
    kl = F.kl_div(s_log, t_prob, reduction="none").sum(dim=-1)
    mask = attention_mask.float()
    return ((kl * mask).sum() / mask.sum().clamp_min(1.0)) * (temperature ** 2)

print("Ternary QAT and Distillation components initialized.")

In [ ]:
# 3. Unified Training Function
def train_model(
    mode="kd_qat",
    num_samples=2000,
    batch_size=8,
    max_length=128,
    epochs=1,
    lr=1e-4,
    alpha=0.5,
    temperature=2.0,
    threshold_factor=0.7,
    save_dir="./model_checkpoints"
):
    out_path = Path(save_dir) / mode
    out_path.mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-135M")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    student = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-135M").to(DEVICE)
    if "qat" in mode:
        replace_linear_layers(student, threshold_factor=threshold_factor)
    student.train()

    teacher = None
    if "kd" in mode:
        teacher = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-360M-Instruct").to(DEVICE)
        teacher.eval()
        for p in teacher.parameters(): p.requires_grad = False

    raw_ds = load_dataset("roneneldan/TinyStories", split=f"train[:{num_samples}]")
    texts = [x["text"] for x in raw_ds if len(x.get("text", "").strip()) > 0]
    
    def collate_fn(batch_texts):
        enc = tokenizer(batch_texts, return_tensors="pt", max_length=max_length, truncation=True, padding="max_length")
        labels = enc["input_ids"].clone()
        labels[enc["attention_mask"] == 0] = -100
        return enc["input_ids"].to(DEVICE), enc["attention_mask"].to(DEVICE), labels.to(DEVICE)

    loader = DataLoader(texts, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    optimizer = torch.optim.AdamW([p for p in student.parameters() if p.requires_grad], lr=lr)

    print(f"--> Training [{mode.upper()}] on {len(texts)} samples ({epochs} epoch(s)) ...")
    history = []
    step = 0
    for ep in range(epochs):
        for input_ids, mask, labels in loader:
            teacher_logits = None
            if teacher is not None:
                with torch.no_grad():
                    teacher_logits = teacher(input_ids=input_ids, attention_mask=mask).logits

            s_out = student(input_ids=input_ids, attention_mask=mask, labels=labels)
            ce = s_out.loss

            if teacher_logits is not None:
                kd = compute_kd_loss(s_out.logits, teacher_logits, mask, temperature=temperature)
                loss = alpha * ce + (1.0 - alpha) * kd
            else:
                kd = torch.tensor(0.0)
                loss = ce

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            optimizer.step()

            step += 1
            history.append({"step": step, "loss": float(loss.item()), "ce": float(ce.item()), "kd": float(kd.item())})
            if step % 50 == 0:
                print(f"  Step {step:04d} | Loss: {loss.item():.4f} | CE: {ce.item():.4f} | KD: {kd.item():.4f}")

    student.save_pretrained(out_path)
    tokenizer.save_pretrained(out_path)
    print(f"--> Completed and saved to {out_path}")
    return student, tokenizer, history

In [ ]:
# 4. Unified Evaluation Function (Perplexity, Memory Footprint, Throughput)
@torch.no_grad()
def evaluate_model(model, tokenizer, is_ternary=False, num_eval_samples=100):
    model.eval()
    if is_ternary:
        # Snap to deployed ternary lattice
        for name, param in model.named_parameters():
            if "weight" in name and param.dim() == 2:
                scale = param.abs().mean().clamp_min(1e-8)
                thresh = 0.7 * scale
                param.copy_(torch.where(param > thresh, scale, torch.where(param < -thresh, -scale, torch.zeros_like(param))))

    val_ds = load_dataset("roneneldan/TinyStories", split=f"validation[:{num_eval_samples}]")
    losses = []
    for sample in val_ds:
        txt = sample["text"]
        if not txt.strip(): continue
        enc = tokenizer(txt, return_tensors="pt", max_length=128, truncation=True).to(DEVICE)
        if enc["input_ids"].shape[1] <= 1: continue
        out = model(**enc, labels=enc["input_ids"])
        losses.append(out.loss.item())

    mean_loss = sum(losses) / len(losses)
    ppl = math.exp(min(mean_loss, 20.0))
    
    # Benchmark generation
    prompt = "Once upon a time, there was a little friendly dog named Spot who"
    inp = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    t0 = time.perf_counter()
    gen = model.generate(**inp, max_new_tokens=40, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    dur = time.perf_counter() - t0
    tok_sec = (gen.shape[1] - inp["input_ids"].shape[1]) / max(dur, 1e-4)
    gen_text = tokenizer.decode(gen[0], skip_special_tokens=True)

    total_p = sum(p.numel() for p in model.parameters())
    size_mb = (total_p * 2 / 1024 / 1024) if not is_ternary else ((total_p * 2 / 8) / 1024 / 1024)

    return {
        "val_loss": round(mean_loss, 4),
        "perplexity": round(ppl, 2),
        "size_mb": round(size_mb, 1),
        "throughput": round(tok_sec, 1),
        "sample": gen_text
    }

In [ ]:
# 5. Run All Factorial Experiments (A, B, C, D)
# Set sample count according to available Kaggle GPU time (e.g. 2000-5000)
SAMPLES = 2000
results = {}

for key, mode, is_ternary in [
    ("Model A (Baseline FP16)", "baseline", False),
    ("Model B (Distilled FP16)", "kd", False),
    ("Model C (Ternary QAT)", "qat", True),
    ("Model D (KD + Ternary QAT)", "kd_qat", True)
]:
    print(f"\n{'='*60}\nStarting: {key}\n{'='*60}")
    model, tok, hist = train_model(mode=mode, num_samples=SAMPLES, epochs=1)
    res = evaluate_model(model, tok, is_ternary=is_ternary)
    res["history"] = hist
    results[key] = res
    print(f"Results for {key}: Loss={res['val_loss']}, PPL={res['perplexity']}, Size={res['size_mb']}MB, Speed={res['throughput']} tok/s")

In [ ]:
# 6. Generate Comparative Markdown & LaTeX Report
import pandas as pd

table_rows = []
for k, v in results.items():
    table_rows.append({
        "Model": k,
        "Precision": "1.58-bit Ternary" if "Ternary" in k else "FP16",
        "Supervision": "KD + CE" if "Distilled" in k or "KD" in k else "CE only",
        "Val Loss": v["val_loss"],
        "Perplexity": v["perplexity"],
        "Size (MB)": v["size_mb"],
        "Speed (tok/s)": v["throughput"]
    })

df = pd.DataFrame(table_rows)
print("\n--- FINAL COMPARATIVE SUMMARY TABLE ---\n")
print(df.to_markdown(index=False))

# Save results
with open("kaggle_final_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nSaved raw metrics to kaggle_final_results.json")

In [ ]:
# 7. Plot Loss Curves & Perplexity vs. Size
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
for k, v in results.items():
    losses = [x["loss"] for x in v["history"]]
    axes[0].plot(losses, label=k, alpha=0.8)
axes[0].set_title("Training Loss Trajectory")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

# Perplexity vs Model Size Bar Chart
names = [k.replace("Model ", "") for k in results.keys()]
ppls = [v["perplexity"] for v in results.values()]
axes[1].bar(names, ppls, color=['#2b5c8f', '#357ebd', '#e26d5c', '#4ba3e3'])
axes[1].set_title("Validation Perplexity Comparison (Lower is Better)")
axes[1].set_ylabel("Perplexity (PPL)")
for i, p in enumerate(ppls):
    axes[1].text(i, p + 0.1, f"{p:.1f}", ha='center', fontweight='bold')
axes[1].grid(axis='y')

plt.tight_layout()
plt.savefig("experiment_comparison.png", dpi=300)
plt.show()
print("Comparison chart saved to experiment_comparison.png")